# 词性标注实践：基于隐马尔可夫模型的方法

隐马尔可夫模型（Hidden Markov Model，HMM）可用如下 5 个参数定义 $\lambda=(A,B,\pi)$：

- $N$：状态数，状态集合 $S=\{s_1,\dots,s_N\}$（对应词性标签表）
- $M$：观测值数，观测集合 $V=\{v_1,\dots,v_M\}$（对应词表）
- $\pi$：初始状态概率
- $A$：状态转移概率矩阵，$a_{ij}=P(q_{t+1}=s_j\mid q_t=s_i)$
- $B$：观测概率矩阵，$b_j(k)=P(o_t=v_k\mid q_t=s_j)$

在词性标注任务中，观测序列 $O$ 对应输入的单词序列 $W$，状态序列 $Q$ 对应每个单词的词性。
本案例将实现：

1. 用**最大似然估计（计数 + 加法平滑）**从已标注语料中学习 $\pi$、$A$、$B$；
2. 用**维特比（Viterbi）算法**在给定观测序列下求解最优词性序列；
3. 复现书中**图 2.4** 给出的例子："上海 浦东 颁布 了 十 件 政策 文件"；
4. 演示**未登录词（OOV）**的处理方式。


## 1. 准备训练语料

构造一个小型的中文词性标注语料，标签体系与书中图 2.4 一致
（NR-专有名词、NN-普通名词、VV-动词、AS-体标记、CD-数词、M-量词，并扩展 JJ-形容词），
用来训练 HMM 的各项参数。


In [1]:
train_corpus = [
    [("上海", "NR"), ("浦东", "NR"), ("颁布", "VV"), ("了", "AS"),
     ("十", "CD"), ("件", "M"), ("政策", "NN"), ("文件", "NN")],
    [("北京", "NR"), ("发布", "VV"), ("了", "AS"),
     ("三", "CD"), ("项", "M"), ("新", "JJ"), ("规定", "NN")],
    [("政府", "NN"), ("颁布", "VV"), ("了", "AS"),
     ("五", "CD"), ("条", "M"), ("法律", "NN")],
    [("上海", "NR"), ("浦东", "NR"), ("建设", "VV"), ("了", "AS"),
     ("一", "CD"), ("座", "M"), ("医院", "NN")],
    [("北京", "NR"), ("政府", "NN"), ("发布", "VV"), ("了", "AS"),
     ("两", "CD"), ("份", "M"), ("文件", "NN")],
    [("浦东", "NR"), ("建设", "VV"), ("了", "AS"),
     ("三", "CD"), ("座", "M"), ("新", "JJ"), ("医院", "NN")],
]

for i, sent in enumerate(train_corpus, 1):
    print(f"句{i}: " + " ".join(f"{w}/{t}" for w, t in sent))


句1: 上海/NR 浦东/NR 颁布/VV 了/AS 十/CD 件/M 政策/NN 文件/NN
句2: 北京/NR 发布/VV 了/AS 三/CD 项/M 新/JJ 规定/NN
句3: 政府/NN 颁布/VV 了/AS 五/CD 条/M 法律/NN
句4: 上海/NR 浦东/NR 建设/VV 了/AS 一/CD 座/M 医院/NN
句5: 北京/NR 政府/NN 发布/VV 了/AS 两/CD 份/M 文件/NN
句6: 浦东/NR 建设/VV 了/AS 三/CD 座/M 新/JJ 医院/NN


## 2. 实现 HMM：参数估计（训练）

给定训练语料，通过**计数 + 加法平滑（Laplace Smoothing）**估计 $\pi$、$A$、$B$：

$$
\pi_i = \frac{\#\{\text{句首标签为 } s_i\} + \alpha}{\#\text{句子数} + \alpha N}
\qquad
A_{ij} = \frac{\#\{s_i \to s_j\} + \alpha}{\#\{\text{从 } s_i \text{ 转出的总次数}\} + \alpha N}
\qquad
B_j(k) = \frac{\#\{s_j \text{ 观测到 } v_k\} + \alpha}{\#\{s_j \text{ 出现总次数}\} + \alpha M}
$$

加法平滑保证了任意状态转移、任意词性-单词组合都不会有零概率——这一点对于维特比算法尤为重要，
因为概率一旦为 0，取对数会得到 $-\infty$，会直接排除该路径。


In [2]:
from collections import defaultdict
import math


class HMMPOSTagger:
    def __init__(self):
        self.tags = None             # 状态集合 S
        self.words = None            # 观测集合 V（词表）
        self.pi = None               # 初始状态概率
        self.A = None                # 转移概率矩阵
        self.B = None                # 观测概率矩阵
        self.open_class_tags = None  # 开放类词性（用于未登录词处理）

    def fit(self, corpus, open_class_tags=("NR", "NN", "VV", "JJ"), alpha=0.1):
        """用最大似然估计（计数 + 加法平滑）从已标注语料中学习模型参数"""
        tag_set = sorted({t for sent in corpus for _, t in sent})
        word_set = sorted({w for sent in corpus for w, _ in sent})
        self.tags = tag_set
        self.words = word_set
        self.open_class_tags = [t for t in open_class_tags if t in tag_set]

        init_count = defaultdict(int)
        trans_count = defaultdict(lambda: defaultdict(int))
        emit_count = defaultdict(lambda: defaultdict(int))
        tag_count = defaultdict(int)

        for sent in corpus:
            tags = [t for _, t in sent]
            init_count[tags[0]] += 1
            for (w, t) in sent:
                emit_count[t][w] += 1
                tag_count[t] += 1
            for i in range(len(tags) - 1):
                trans_count[tags[i]][tags[i + 1]] += 1

        n_sent, n_tag, n_word = len(corpus), len(tag_set), len(word_set)

        # 初始概率 pi
        self.pi = {t: (init_count[t] + alpha) / (n_sent + alpha * n_tag) for t in tag_set}

        # 转移概率 A
        self.A = {}
        for t1 in tag_set:
            denom = sum(trans_count[t1][t2] for t2 in tag_set) + alpha * n_tag
            self.A[t1] = {t2: (trans_count[t1][t2] + alpha) / denom for t2 in tag_set}

        # 观测概率 B（为未登录词额外保留一部分概率质量）
        self.B = {}
        for t in tag_set:
            denom = tag_count[t] + alpha * n_word
            self.B[t] = {w: (emit_count[t][w] + alpha) / denom for w in word_set}
            self.B[t]["<UNK>"] = alpha / denom
        return self

    def _emit_prob(self, tag, word):
        """未登录词处理：仅在开放类词性（名词、动词、形容词等）上给予概率，
        封闭类词性（如 AS、CD、M 等功能词）上未登录词概率视为极小值"""
        if word in self.B[tag]:
            return self.B[tag][word]
        if tag in self.open_class_tags:
            return self.B[tag]["<UNK>"]
        return 1e-12

    def viterbi(self, words, verbose=False):
        """维特比算法：在对数空间中动态规划求解最优词性序列"""
        n = len(words)
        tags = self.tags
        dp = [dict() for _ in range(n)]
        backpointer = [dict() for _ in range(n)]

        for t in tags:
            emit_p = self._emit_prob(t, words[0])
            dp[0][t] = math.log(self.pi[t]) + math.log(emit_p)
            backpointer[0][t] = None

        for i in range(1, n):
            for t in tags:
                emit_p = self._emit_prob(t, words[i])
                best_prev, best_score = None, float("-inf")
                for t_prev in tags:
                    score = dp[i - 1][t_prev] + math.log(self.A[t_prev][t])
                    if score > best_score:
                        best_score, best_prev = score, t_prev
                dp[i][t] = best_score + math.log(emit_p)
                backpointer[i][t] = best_prev

        best_last_tag = max(tags, key=lambda t: dp[n - 1][t])
        path = [best_last_tag]
        for i in range(n - 1, 0, -1):
            path.append(backpointer[i][path[-1]])
        path.reverse()

        if verbose:
            for i, w in enumerate(words):
                mark = " (未登录词)" if w not in self.words else ""
                print(f"  位置{i+1}: {w}{mark} -> {path[i]}")
        return path


hmm = HMMPOSTagger().fit(train_corpus)
print("状态集合 S (词性):", hmm.tags)
print("词表大小 |V| =", len(hmm.words))


状态集合 S (词性): ['AS', 'CD', 'JJ', 'M', 'NN', 'NR', 'VV']
词表大小 |V| = 24


## 3. 查看训练得到的模型参数

In [3]:
print("初始概率 π：")
for t in hmm.tags:
    print(f"  π({t:>2s}) = {hmm.pi[t]:.4f}")

print("\n转移概率 A（部分，从状态 VV 出发）：")
for t2 in hmm.tags:
    print(f"  A(VV -> {t2:>2s}) = {hmm.A['VV'][t2]:.4f}")

print("\n观测概率 B（部分，状态 NN 下若干高频词）：")
top_words = sorted(hmm.B["NN"].items(), key=lambda kv: -kv[1])[:6]
for w, p in top_words:
    print(f"  B(NN 观测到 {w!r}) = {p:.4f}")


初始概率 π：
  π(AS) = 0.0149
  π(CD) = 0.0149
  π(JJ) = 0.0149
  π( M) = 0.0149
  π(NN) = 0.1642
  π(NR) = 0.7612
  π(VV) = 0.0149

转移概率 A（部分，从状态 VV 出发）：
  A(VV -> AS) = 0.9104
  A(VV -> CD) = 0.0149
  A(VV -> JJ) = 0.0149
  A(VV ->  M) = 0.0149
  A(VV -> NN) = 0.0149
  A(VV -> NR) = 0.0149
  A(VV -> VV) = 0.0149

观测概率 B（部分，状态 NN 下若干高频词）：
  B(NN 观测到 '医院') = 0.1842
  B(NN 观测到 '政府') = 0.1842
  B(NN 观测到 '文件') = 0.1842
  B(NN 观测到 '政策') = 0.0965
  B(NN 观测到 '法律') = 0.0965
  B(NN 观测到 '规定') = 0.0965


## 4. 复现书中图 2.4 的例子

书中图 2.4 给出的例子是：

> 上海 浦东 颁布 了 十 件 政策 文件
> NR&nbsp;&nbsp;&nbsp; NR&nbsp;&nbsp;&nbsp; VV&nbsp;&nbsp;&nbsp; AS&nbsp;&nbsp;&nbsp; CD&nbsp;&nbsp;&nbsp; M&nbsp;&nbsp;&nbsp; NN&nbsp;&nbsp;&nbsp; NN

下面用训练好的 HMM，通过维特比算法对这句话进行解码，并与标准答案比对。


In [4]:
test1 = ["上海", "浦东", "颁布", "了", "十", "件", "政策", "文件"]
gold1 = ["NR", "NR", "VV", "AS", "CD", "M", "NN", "NN"]

print("=== 复现图 2.4 例句 ===")
pred1 = hmm.viterbi(test1, verbose=True)
print("\n预测结果:", pred1)
print("标准答案:", gold1)
print("是否完全一致:", pred1 == gold1)


=== 复现图 2.4 例句 ===
  位置1: 上海 -> NR
  位置2: 浦东 -> NR
  位置3: 颁布 -> VV
  位置4: 了 -> AS
  位置5: 十 -> CD
  位置6: 件 -> M
  位置7: 政策 -> NN
  位置8: 文件 -> NN

预测结果: ['NR', 'NR', 'VV', 'AS', 'CD', 'M', 'NN', 'NN']
标准答案: ['NR', 'NR', 'VV', 'AS', 'CD', 'M', 'NN', 'NN']
是否完全一致: True


## 5. 未登录词（OOV）处理演示

原文指出，实际应用 HMM 进行词性标注时通常需要解决**未登录词**问题：
由于观测概率矩阵 $B$ 中不存在该词，需要进行特殊处理。本案例采用原文提到的思路之一——
"在观测概率矩阵中设置该词以同样的概率观察到所有标记类别"的改良版：
**只在开放类词性（名词、动词、形容词等）上给予未登录词概率，封闭类（功能词）上概率极低**，
这样即便某个词从未在训练语料中出现，模型仍可以**主要依靠上下文的转移概率**对其词性做出合理推断。

下面用一个包含未登录词 **"深圳"** 的新句子进行测试（"深圳"未出现在训练语料中）。


In [5]:
test2 = ["深圳", "颁布", "了", "一", "项", "政策"]

print("=== 未登录词演示: 深圳 颁布 了 一 项 政策 ===")
pred2 = hmm.viterbi(test2, verbose=True)
print("\n预测结果:", pred2)
print("\n可以看到“深圳”虽是未登录词，但由于")
print("  (1) 只在开放类词性上保留了非零的观测概率，")
print("  (2) 句首状态更倾向于 NR（初始概率 π(NR) 较高），")
print("模型依然能够依靠上下文正确地将其推断为专有名词 NR。")


=== 未登录词演示: 深圳 颁布 了 一 项 政策 ===
  位置1: 深圳 (未登录词) -> NR
  位置2: 颁布 -> VV
  位置3: 了 -> AS
  位置4: 一 -> CD
  位置5: 项 -> M
  位置6: 政策 -> NN

预测结果: ['NR', 'VV', 'AS', 'CD', 'M', 'NN']

可以看到“深圳”虽是未登录词，但由于
  (1) 只在开放类词性上保留了非零的观测概率，
  (2) 句首状态更倾向于 NR（初始概率 π(NR) 较高），
模型依然能够依靠上下文正确地将其推断为专有名词 NR。


### 对比：若不区分开放类/封闭类词性

作为对照，我们再实现一版"最简单粗暴"的未登录词处理——对所有词性都赋予相同的未登录词概率
（对应原文中提到的"较为粗糙"的第一种做法），观察其对结果的影响。


In [6]:
class HMMPOSTaggerNaiveUNK(HMMPOSTagger):
    """未登录词处理的粗糙版本：所有词性上一视同仁地赋予相同的未登录词概率"""
    def _emit_prob(self, tag, word):
        if word in self.B[tag]:
            return self.B[tag][word]
        return self.B[tag]["<UNK>"]


hmm_naive = HMMPOSTaggerNaiveUNK().fit(train_corpus)

print("=== 粗糙版未登录词处理（所有词性一视同仁）结果 ===")
pred2_naive = hmm_naive.viterbi(test2, verbose=True)
print("\n改良版（区分开放类/封闭类）预测:", pred2)
print("粗糙版（不区分）预测:          ", pred2_naive)


=== 粗糙版未登录词处理（所有词性一视同仁）结果 ===
  位置1: 深圳 (未登录词) -> NR
  位置2: 颁布 -> VV
  位置3: 了 -> AS
  位置4: 一 -> CD
  位置5: 项 -> M
  位置6: 政策 -> NN

改良版（区分开放类/封闭类）预测: ['NR', 'VV', 'AS', 'CD', 'M', 'NN']
粗糙版（不区分）预测:           ['NR', 'VV', 'AS', 'CD', 'M', 'NN']


可以看到两种处理方式在本例上可能给出相同或不同的结果，这取决于具体的概率数值。
但从原理上说，"不区分开放类/封闭类"的粗糙做法允许未登录词被标注为任何词性（包括 AS、CD、M 等功能词），
这在真实场景中通常是不合理的——如原文所说，未登录词绝大多数属于开放类词汇（名词、动词、形容词等），
区分处理能让模型的先验假设更符合语言学事实。


## 6. 附：长句截断问题（简要说明）

原文还提到，在实际应用中使用 HMM 进行词性标注时，还需要处理**长句子**的问题——
如《人民日报》语料中有些句子长度甚至超过 120 个字，过长的句子会显著增加维特比算法的计算量
（复杂度为 $O(nL^2)$，$n$ 为句子长度，$L$ 为词性数量），因此通常会**限定句子的最大长度**，
超过限制时在最近的标点处进行截断。本案例语料规模较小，未展示这一处理逻辑，读者可自行在
`viterbi` 函数外层包装一个按最大长度截断句子的预处理步骤。


## 7. 小结

本 Notebook 完整实现并验证了 2.4.2 节介绍的基于隐马尔可夫模型的词性标注方法：

1. 用**计数 + 加法平滑**从标注语料中估计 $\pi$、$A$、$B$ 三组参数；
2. 用**维特比算法**在对数空间高效求解最优词性序列，成功复现了书中图 2.4 的例子；
3. 演示了**未登录词处理**的两种思路，并说明了"区分开放类/封闭类词性"这一改良思路的合理性。

与 2.4.1 节的规则方法相比，HMM 方法的优势在于能够**自动从语料中学习统计规律**，
不需要人工设计规则模板；但它依赖**一阶马尔可夫假设**（当前标签只与前一个标签有关），
对更复杂的长距离依赖关系建模能力有限，这也是后续章节中基于条件随机场、神经网络等方法要解决的问题。
